# Mental Health BERT Fine-Tuned – Kaggle Evaluation Notebook

In [1]:
pip install transformers datasets scikit-learn --quiet

In [2]:
"""from google.colab import files
uploaded = files.upload()"""

'from google.colab import files\nuploaded = files.upload()'

In [3]:
"""import zipfile
import os

zip_file = "archive.zip"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Files in dataset folder:")
print(os.listdir("dataset"))
"""

'import zipfile\nimport os\n\nzip_file = "archive.zip"\n\nwith zipfile.ZipFile(zip_file, \'r\') as zip_ref:\n    zip_ref.extractall("dataset")\n\nprint("Files in dataset folder:")\nprint(os.listdir("dataset"))\n'

In [4]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup

In [6]:
train_df = pd.read_csv("/content/mental_heath_unbanlanced.csv")
test_df = pd.read_csv("/content/mental_health_combined_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (49612, 3)
Test shape: (992, 2)


,Unique_ID,text,status
0,0.0,oh my gosh,Anxiety
1,1.0,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2.0,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3.0,I've shifted my focus to something else but I'...,Anxiety
4,4.0,"I'm restless and restless, it's been a month n...",Anxiety


In [7]:
train_df = train_df[['text','status']]
test_df = test_df[['text','status']]

train_df.dropna(inplace=True)
test_df.dropna(inplace=True)

print(train_df.head())

                                                text   status
0                                         oh my gosh  Anxiety
1  trouble sleeping, confused mind, restless hear...  Anxiety
2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3  I've shifted my focus to something else but I'...  Anxiety
4  I'm restless and restless, it's been a month n...  Anxiety


In [8]:
label_encoder = LabelEncoder()

train_df['label'] = label_encoder.fit_transform(train_df['status'])
test_df['label'] = label_encoder.transform(test_df['status'])

num_labels = len(train_df['label'].unique())

print("Number of classes:", num_labels)

Number of classes: 4


In [9]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'],
    train_df['label'],
    test_size=0.1,
    random_state=42
)

In [10]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
class MentalHealthDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [12]:
train_dataset = MentalHealthDataset(train_texts, train_labels, tokenizer)
val_dataset = MentalHealthDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [13]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [14]:
optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * 3

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [15]:
from tqdm import tqdm

epochs = 5

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Loss:", avg_loss)

100%|██████████| 2791/2791 [17:18<00:00,  2.69it/s]


Epoch 1 Loss: 0.4696972341183606


100%|██████████| 2791/2791 [17:27<00:00,  2.66it/s]


Epoch 2 Loss: 0.3023449744974486


100%|██████████| 2791/2791 [17:28<00:00,  2.66it/s]


Epoch 3 Loss: 0.20472029566034722


100%|██████████| 2791/2791 [17:27<00:00,  2.66it/s]


Epoch 4 Loss: 0.1649169283837283


100%|██████████| 2791/2791 [17:29<00:00,  2.66it/s]

Epoch 5 Loss: 0.16538445227580903


In [16]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)

        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

In [17]:
print(classification_report(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.86      0.86      0.86       532
           1       0.79      0.77      0.78      1488
           2       0.96      0.96      0.96      1826
           3       0.75      0.78      0.77      1116

    accuracy                           0.85      4962
   macro avg       0.84      0.84      0.84      4962
weighted avg       0.85      0.85      0.85      4962



In [18]:
model.save_pretrained("mental_health_bert_model")
tokenizer.save_pretrained("mental_health_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('mental_health_bert_model/tokenizer_config.json',
 'mental_health_bert_model/tokenizer.json')

In [72]:
!pip install gradio transformers torch --quiet

In [73]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "mental_health_bert_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

labels = ["Anxiety", "Depression", "Normal", "Suicidal"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [74]:
import re

def predict(text):

    if text is None or text.strip() == "":
        return "Please enter text", {}

    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.lower().strip()

    encoding = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

    probs = F.softmax(outputs.logits, dim=1)[0].cpu()

    pred = torch.argmax(probs).item()
    confidence = probs[pred].item() * 100

    result = {labels[i]: float(probs[i]) for i in range(len(labels))}

    message = f"Prediction: {labels[pred]} | Confidence: {confidence:.2f}%"

    return message, result

In [75]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🧠 AI Mental Health Detection System
    ### BERT-based NLP Model

    Enter a sentence to analyze emotional mental health condition.
    """)

    with gr.Row():

        with gr.Column(scale=1):

            text_input = gr.Textbox(
                label="Enter Text",
                placeholder="Example: I feel very stressed and anxious today...",
                lines=4
            )

            analyze_btn = gr.Button("Analyze Mental Health")

        with gr.Column(scale=1):

            prediction_output = gr.Textbox(label="Prediction Result")

            probability_output = gr.Label(label="Class Probabilities")

    analyze_btn.click(
        fn=predict,
        inputs=text_input,
        outputs=[prediction_output, probability_output]
    )

    gr.Markdown("""
    ### Mental Health Categories

    🟡 **Anxiety** – Stress, fear, excessive worry
    🔵 **Depression** – Sadness, hopelessness
    🟢 **Normal** – Healthy emotional state
    🔴 **Suicidal** – Self-harm related thoughts
    """)

demo.launch(share=True)

/tmp/ipykernel_218/2248572883.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b34e08c3108977fdab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
